# 05 RQ4 Explainability Analysis Under Real-World Conditions

This notebook evaluates Grad-CAM explanations under real-world conditions.

It performs:
- checkpoint loading
- Grad-CAM generation
- representative correct and incorrect case analysis
- leaf-focused attention assessment
- comparison of explanation behavior for correct and incorrect predictions
- export of tables and figures

Outputs:
- Table 7: explainability summary under real-world evaluation
- Table 8: explanation behavior for correct and incorrect predictions
- Figure 7: representative Grad-CAM explanations
- Figure 8: comparison of explanation behavior in reliable and unreliable predictions
- ZIP archive of RQ4 outputs

In [1]:
# ----------------------------------------
# Section 1: Imports
# ----------------------------------------

import os
import json
import random
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

In [2]:
# ----------------------------------------
# Section 2: Reproducibility setup
# ----------------------------------------

SEED = 42

def seed_everything(seed: int = 42) -> None:
    """
    Set random seeds for reproducibility across Python, NumPy, and PyTorch.
    """
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id: int) -> None:
    """
    Ensure each DataLoader worker uses a deterministic seed.
    """
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

seed_everything(SEED)

print("Reproducibility setup completed")
print(f"Global seed: {SEED}")

Reproducibility setup completed
Global seed: 42


In [3]:
# ----------------------------------------
# Section 3: Configuration
# ----------------------------------------

CONFIG = {
    "seed": SEED,
    "image_size": 224,
    "batch_size": 32,
    "num_workers": 0,
    "plantdoc_root": "/kaggle/input/datasets/thedataeng/plantdoc",

    # Update this if your Kaggle dataset input name differs
    "checkpoint_root": "/kaggle/input/datasets/thedataeng/thesis-train-backbones-outputs",

    "model_names": ["resnet50", "efficientnet_b0", "mobilenet_v2"],
    "figure7_models": ["resnet50", "efficientnet_b0", "mobilenet_v2"],
    "output_root": "/kaggle/working/thesis_outputs/rq4_explainability",
    "audit_batches": 8,
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_ROOT = Path(CONFIG["output_root"])
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

PRETTY_NAMES = {
    "resnet50": "ResNet50",
    "efficientnet_b0": "EfficientNet-B0",
    "mobilenet_v2": "MobileNetV2",
}

print("Configuration loaded")
print(f"Device: {DEVICE}")
print(f"Models: {CONFIG['model_names']}")
print(f"Output root: {OUTPUT_ROOT}")

Configuration loaded
Device: cuda
Models: ['resnet50', 'efficientnet_b0', 'mobilenet_v2']
Output root: /kaggle/working/thesis_outputs/rq4_explainability


In [5]:
# ----------------------------------------
# Section 4: Helper functions
# ----------------------------------------

def ensure_dir(path: Path) -> Path:
    """
    Create a directory if it does not exist and return the Path object.
    """
    path.mkdir(parents=True, exist_ok=True)
    return path

def get_eval_transform(image_size: int = 224):
    """
    Create the evaluation transform used across RQ4 experiments.
    """
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
    ])

def create_model(model_name: str, num_classes: int):
    """
    Create the selected backbone model and replace the classification head.
    Returns the model and the target layer used for Grad-CAM.
    """
    if model_name == "resnet50":
        model = models.resnet50(weights=None)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        target_layer = model.layer4[-1].conv3

    elif model_name == "efficientnet_b0":
        model = models.efficientnet_b0(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
        target_layer = model.features[-1][0]

    elif model_name == "mobilenet_v2":
        model = models.mobilenet_v2(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
        target_layer = model.features[-1]

    else:
        raise ValueError(f"Unsupported model name: {model_name}")

    return model.to(DEVICE), target_layer

def checkpoint_path(model_name: str) -> Path:
    """
    Return the checkpoint path for the selected model.
    """
    return Path(CONFIG["checkpoint_root"]) / "checkpoints" / f"{model_name}_seed{SEED}_best.pt"

def save_table(df: pd.DataFrame, name: str) -> None:
    """
    Save a DataFrame as CSV in the tables directory.
    """
    table_dir = ensure_dir(OUTPUT_ROOT / "tables")
    csv_path = table_dir / f"{name}.csv"
    df.to_csv(csv_path, index=False)
    print(f"Saved table: {csv_path}")

def save_figure(fig: plt.Figure, name: str) -> None:
    """
    Save a matplotlib figure as PDF in the figures directory.
    """
    fig_dir = ensure_dir(OUTPUT_ROOT / "figures")
    pdf_path = fig_dir / f"{name}.pdf"
    fig.tight_layout()
    fig.savefig(pdf_path, format="pdf", bbox_inches="tight")
    plt.close(fig)
    print(f"Saved figure: {pdf_path}")

def pretty_metric(x: float) -> float:
    """
    Round a metric value for cleaner table presentation.
    """
    return round(float(x), 4)

def denormalize(image_tensor: torch.Tensor) -> torch.Tensor:
    """
    Convert a normalized image tensor back to the [0, 1] range for visualization.
    """
    mean = torch.tensor([0.485, 0.456, 0.406], device=image_tensor.device).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=image_tensor.device).view(3, 1, 1)
    return torch.clamp(image_tensor * std + mean, 0, 1)

def simple_leaf_mask(image_np: np.ndarray) -> np.ndarray:
    """
    Create a simple foreground mask that approximates the leaf region using RGB conditions.
    This is a heuristic foreground mask, not a ground-truth segmentation.
    """
    r = image_np[:, :, 0]
    g = image_np[:, :, 1]
    b = image_np[:, :, 2]

    mask = (g > r * 0.85) & (g > b * 0.85) & (g > 0.2)
    return mask.astype(np.uint8)

def overlay_cam(image_np: np.ndarray, cam_np: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    """
    Overlay a Grad-CAM heatmap on top of an RGB image.
    """
    cmap = plt.get_cmap("jet")
    heat = cmap(cam_np)[..., :3]
    overlay = (1 - alpha) * image_np + alpha * heat
    return np.clip(overlay, 0, 1)

print('Done')

Done


In [6]:
# ----------------------------------------
# Section 5: Grad-CAM implementation
# ----------------------------------------

class GradCAM:
    """
    Grad-CAM implementation for CNN-based image classification models.
    """

    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None

        self.forward_handle = target_layer.register_forward_hook(self.forward_hook)
        self.backward_handle = target_layer.register_full_backward_hook(self.backward_hook)

    def forward_hook(self, module, inputs, outputs):
        """
        Store forward activations from the target layer.
        """
        self.activations = outputs.detach()

    def backward_hook(self, module, grad_input, grad_output):
        """
        Store gradients from the target layer during backpropagation.
        """
        self.gradients = grad_output[0].detach()

    def __call__(self, x: torch.Tensor, class_idx: torch.Tensor = None):
        """
        Generate Grad-CAM heatmaps for the provided batch.
        """
        self.model.zero_grad(set_to_none=True)

        logits = self.model(x)

        if class_idx is None:
            class_idx = logits.argmax(dim=1)

        score = logits[torch.arange(logits.shape[0]), class_idx].sum()
        score.backward()

        grads = self.gradients
        acts = self.activations

        weights = grads.mean(dim=(2, 3), keepdim=True)
        cam = (weights * acts).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)
        cam = cam.squeeze(1)

        cam = cam - cam.amin(dim=(1, 2), keepdim=True)
        cam = cam / (cam.amax(dim=(1, 2), keepdim=True) + 1e-8)

        return cam.detach(), logits.detach()

    def close(self) -> None:
        """
        Remove all registered hooks.
        """
        self.forward_handle.remove()
        self.backward_handle.remove()

print('Done')

Done


In [7]:
# ----------------------------------------
# Section 6: Dataset loading
# ----------------------------------------

pd_root = Path(CONFIG["plantdoc_root"])
assert pd_root.exists(), f"Missing PlantDoc directory: {pd_root}"

eval_tfms = get_eval_transform(CONFIG["image_size"])

plantdoc_dataset = datasets.ImageFolder(pd_root, transform=eval_tfms)

generator = torch.Generator()
generator.manual_seed(SEED)

plantdoc_loader = DataLoader(
    plantdoc_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    worker_init_fn=seed_worker,
    generator=generator,
    pin_memory=True,
)

CLASS_NAMES = plantdoc_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)

print("PlantDoc dataset loaded successfully")
print(f"PlantDoc samples:   {len(plantdoc_dataset)}")
print(f"Number of classes:  {NUM_CLASSES}")

PlantDoc dataset loaded successfully
PlantDoc samples:   2555
Number of classes:  27


In [8]:
# ----------------------------------------
# Section 7: Checkpoint loading
# ----------------------------------------

models_loaded = {}
target_layers = {}

for idx, model_name in enumerate(CONFIG["model_names"], start=1):
    print(f"Loading checkpoint {idx}/{len(CONFIG['model_names'])}: {model_name}")

    ckpt_path = checkpoint_path(model_name)
    assert ckpt_path.exists(), f"Missing checkpoint: {ckpt_path}"

    model, target_layer = create_model(model_name, NUM_CLASSES)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()

    models_loaded[model_name] = model
    target_layers[model_name] = target_layer

    print(f"Loaded checkpoint: {ckpt_path}")

print("All checkpoints loaded successfully")

Loading checkpoint 1/3: resnet50
Loaded checkpoint: /kaggle/input/datasets/thedataeng/thesis-train-backbones-outputs/checkpoints/resnet50_seed42_best.pt
Loading checkpoint 2/3: efficientnet_b0
Loaded checkpoint: /kaggle/input/datasets/thedataeng/thesis-train-backbones-outputs/checkpoints/efficientnet_b0_seed42_best.pt
Loading checkpoint 3/3: mobilenet_v2
Loaded checkpoint: /kaggle/input/datasets/thedataeng/thesis-train-backbones-outputs/checkpoints/mobilenet_v2_seed42_best.pt
All checkpoints loaded successfully


In [9]:
# ----------------------------------------
# Section 8: Representative Grad-CAM panel preparation
# ----------------------------------------

figure7_examples = {}

images, labels = next(iter(plantdoc_loader))
images = images.to(DEVICE)

for model_name in CONFIG["figure7_models"]:
    print(f"Preparing representative Grad-CAM examples for: {model_name}")

    model = models_loaded[model_name]
    target_layer = target_layers[model_name]
    cam_engine = GradCAM(model, target_layer)

    cams, logits = cam_engine(images)
    probs = torch.softmax(logits, dim=1)
    preds = probs.argmax(dim=1)
    conf = probs.max(dim=1).values

    correct_idx = None
    wrong_idx = None

    for i in range(len(images)):
        if preds[i].item() == labels[i].item() and correct_idx is None:
            correct_idx = i
        if preds[i].item() != labels[i].item() and wrong_idx is None:
            wrong_idx = i

    if correct_idx is None:
        correct_idx = 0

    if wrong_idx is None:
        wrong_idx = min(1, len(images) - 1)

    figure7_examples[model_name] = {
        "cams": cams.detach().cpu(),
        "preds": preds.detach().cpu(),
        "conf": conf.detach().cpu(),
        "correct_idx": correct_idx,
        "wrong_idx": wrong_idx,
    }

    cam_engine.close()

print("Representative Grad-CAM examples prepared successfully")

Preparing representative Grad-CAM examples for: resnet50
Preparing representative Grad-CAM examples for: efficientnet_b0
Preparing representative Grad-CAM examples for: mobilenet_v2
Representative Grad-CAM examples prepared successfully


In [10]:
# ----------------------------------------
# Section 9: Figure 7 - Representative Grad-CAM explanations
# ----------------------------------------

for model_name in CONFIG["figure7_models"]:
    print(f"Creating representative Grad-CAM figure for: {model_name}")

    example = figure7_examples[model_name]

    fig, axes = plt.subplots(2, 2, figsize=(8.5, 8.5))

    selected_indices = [example["correct_idx"], example["wrong_idx"]]

    for row, idx in enumerate(selected_indices):
        image_np = denormalize(images[idx]).detach().cpu().permute(1, 2, 0).numpy()
        cam_np = example["cams"][idx].numpy()

        axes[row, 0].imshow(image_np)
        axes[row, 0].axis("off")
        axes[row, 0].set_title(
            f"Original image\nTrue: {CLASS_NAMES[labels[idx]]}"
        )

        axes[row, 1].imshow(overlay_cam(image_np, cam_np))
        axes[row, 1].axis("off")
        axes[row, 1].set_title(
            f"Grad-CAM overlay\nPred: {CLASS_NAMES[example['preds'][idx]]} | p={example['conf'][idx].item():.2f}"
        )

    fig.suptitle(
        f"Figure 7. Representative Grad-CAM Explanations Under Real-World Conditions ({PRETTY_NAMES[model_name]})",
        y=1.02
    )

    safe_model_name = PRETTY_NAMES[model_name].replace("-", "").replace(" ", "_")
    save_figure(fig, f"Figure_7_Representative_GradCAM_Explanations_{safe_model_name}")

print("All Figure 7 Grad-CAM example figures saved successfully")

Creating representative Grad-CAM figure for: resnet50
Saved figure: /kaggle/working/thesis_outputs/rq4_explainability/figures/Figure_7_Representative_GradCAM_Explanations_ResNet50.pdf
Creating representative Grad-CAM figure for: efficientnet_b0
Saved figure: /kaggle/working/thesis_outputs/rq4_explainability/figures/Figure_7_Representative_GradCAM_Explanations_EfficientNetB0.pdf
Creating representative Grad-CAM figure for: mobilenet_v2
Saved figure: /kaggle/working/thesis_outputs/rq4_explainability/figures/Figure_7_Representative_GradCAM_Explanations_MobileNetV2.pdf
All Figure 7 Grad-CAM example figures saved successfully


In [11]:
# ----------------------------------------
# Section 10: Explanation behavior audit for all models
# ----------------------------------------

audit_rows = []

for model_idx, model_name in enumerate(CONFIG["model_names"], start=1):
    print(f"Running explanation audit for model {model_idx}/{len(CONFIG['model_names'])}: {model_name}")

    model = models_loaded[model_name]
    target_layer = target_layers[model_name]
    cam_engine = GradCAM(model, target_layer)

    batch_counter = 0

    for batch_images, batch_labels in plantdoc_loader:
        batch_images = batch_images.to(DEVICE)

        cams_batch, logits_batch = cam_engine(batch_images)
        probs_batch = torch.softmax(logits_batch, dim=1)
        preds_batch = probs_batch.argmax(dim=1)
        conf_batch = probs_batch.max(dim=1).values

        for i in range(batch_images.size(0)):
            image_np = denormalize(batch_images[i]).detach().cpu().permute(1, 2, 0).numpy()
            cam_np = cams_batch[i].detach().cpu().numpy()

            leaf_mask = simple_leaf_mask(image_np)
            leaf_focus = float((cam_np * leaf_mask).sum() / (cam_np.sum() + 1e-8))
            bg_leakage = 1.0 - leaf_focus

            audit_rows.append({
                "Model": model_name,
                "TrueClass": CLASS_NAMES[batch_labels[i].item()],
                "PredictedClass": CLASS_NAMES[preds_batch[i].item()],
                "Confidence": pretty_metric(conf_batch[i].item()),
                "LeafFocusRatio": pretty_metric(leaf_focus),
                "BackgroundLeakage": pretty_metric(bg_leakage),
                "Correct": int(preds_batch[i].item() == batch_labels[i].item()),
            })

        batch_counter += 1
        print(f"  Processed batch {batch_counter}/{CONFIG['audit_batches']}")

        if batch_counter >= CONFIG["audit_batches"]:
            break

    cam_engine.close()

audit_df = pd.DataFrame(audit_rows)
audit_df["Model"] = audit_df["Model"].map(PRETTY_NAMES)

print("Explanation audit completed successfully for all models")
display(audit_df.head(10))

Running explanation audit for model 1/3: resnet50
  Processed batch 1/8
  Processed batch 2/8
  Processed batch 3/8
  Processed batch 4/8
  Processed batch 5/8
  Processed batch 6/8
  Processed batch 7/8
  Processed batch 8/8
Running explanation audit for model 2/3: efficientnet_b0
  Processed batch 1/8
  Processed batch 2/8
  Processed batch 3/8
  Processed batch 4/8
  Processed batch 5/8
  Processed batch 6/8
  Processed batch 7/8
  Processed batch 8/8
Running explanation audit for model 3/3: mobilenet_v2
  Processed batch 1/8
  Processed batch 2/8
  Processed batch 3/8
  Processed batch 4/8
  Processed batch 5/8
  Processed batch 6/8
  Processed batch 7/8
  Processed batch 8/8
Explanation audit completed successfully for all models


,Model,TrueClass,PredictedClass,Confidence,LeafFocusRatio,BackgroundLeakage,Correct
0,ResNet50,Apple Scab Leaf,Tomato Early blight leaf,0.3796,1.0000,0.0000,0
1,ResNet50,Apple Scab Leaf,Cherry leaf,0.9015,0.7383,0.2617,0
2,ResNet50,Apple Scab Leaf,Corn leaf blight,0.9988,0.4470,0.5530,0
3,ResNet50,Apple Scab Leaf,Tomato Early blight leaf,0.6513,0.6612,0.3388,0
4,ResNet50,Apple Scab Leaf,Potato leaf early blight,0.4208,0.9619,0.0381,0
5,ResNet50,Apple Scab Leaf,Cherry leaf,0.7657,0.9616,0.0384,0
6,ResNet50,Apple Scab Leaf,Cherry leaf,0.9125,0.8940,0.1060,0
7,ResNet50,Apple Scab Leaf,Apple rust leaf,0.8620,0.8439,0.1561,0
8,ResNet50,Apple Scab Leaf,Bell_pepper leaf spot,0.8700,0.8520,0.1480,0
9,ResNet50,Apple Scab Leaf,Cherry leaf,0.4828,0.8298,0.1702,0


In [12]:
# ----------------------------------------
# Section 11: Save audit samples
# ----------------------------------------

for model_name in PRETTY_NAMES.values():
    sample_audit_df = audit_df[audit_df["Model"] == model_name].head(100).copy()
    save_table(sample_audit_df, f"Optional_Explanation_Audit_First_100_{model_name}")

print("Explanation audit samples saved successfully")

Saved table: /kaggle/working/thesis_outputs/rq4_explainability/tables/Optional_Explanation_Audit_First_100_ResNet50.csv
Saved table: /kaggle/working/thesis_outputs/rq4_explainability/tables/Optional_Explanation_Audit_First_100_EfficientNet-B0.csv
Saved table: /kaggle/working/thesis_outputs/rq4_explainability/tables/Optional_Explanation_Audit_First_100_MobileNetV2.csv
Explanation audit samples saved successfully


In [13]:
# ----------------------------------------
# Section 12: Save Table 7 - Explainability summary
# ----------------------------------------

table7_df = (
    audit_df.groupby("Model")
    .agg(
        Avg_Confidence=("Confidence", "mean"),
        Avg_Leaf_Focus_Ratio=("LeafFocusRatio", "mean"),
        Avg_Background_Leakage=("BackgroundLeakage", "mean"),
        Correct_Prediction_Rate=("Correct", "mean"),
    )
    .reset_index()
)

table7_df["Avg_Confidence"] = table7_df["Avg_Confidence"].apply(pretty_metric)
table7_df["Avg_Leaf_Focus_Ratio"] = table7_df["Avg_Leaf_Focus_Ratio"].apply(pretty_metric)
table7_df["Avg_Background_Leakage"] = table7_df["Avg_Background_Leakage"].apply(pretty_metric)
table7_df["Correct_Prediction_Rate"] = table7_df["Correct_Prediction_Rate"].apply(pretty_metric)

save_table(table7_df, "Table_7_Explainability_Summary_RealWorld")

print("Table 7 saved successfully")
display(table7_df)

Saved table: /kaggle/working/thesis_outputs/rq4_explainability/tables/Table_7_Explainability_Summary_RealWorld.csv
Table 7 saved successfully


,Model,Avg_Confidence,Avg_Leaf_Focus_Ratio,Avg_Background_Leakage,Correct_Prediction_Rate
0,EfficientNet-B0,0.7086,0.8521,0.1479,0.1094
1,MobileNetV2,0.5639,0.8512,0.1488,0.0664
2,ResNet50,0.7072,0.8373,0.1627,0.1016


In [14]:
# ----------------------------------------
# Section 13: Save Table 8 - Correct vs incorrect explanation behavior
# ----------------------------------------

table8_rows = []

for model_name in PRETTY_NAMES.values():
    model_df = audit_df[audit_df["Model"] == model_name].copy()

    correct_df = model_df[model_df["Correct"] == 1].copy()
    incorrect_df = model_df[model_df["Correct"] == 0].copy()

    table8_rows.append({
        "Model": model_name,
        "Prediction Type": "Correct Predictions",
        "Avg. Confidence": pretty_metric(correct_df["Confidence"].mean()) if len(correct_df) > 0 else np.nan,
        "Avg. Leaf Focus Ratio": pretty_metric(correct_df["LeafFocusRatio"].mean()) if len(correct_df) > 0 else np.nan,
        "Avg. Background Leakage": pretty_metric(correct_df["BackgroundLeakage"].mean()) if len(correct_df) > 0 else np.nan,
        "Interpretation": "More concentrated attention on relevant leaf regions",
    })

    table8_rows.append({
        "Model": model_name,
        "Prediction Type": "Incorrect Predictions",
        "Avg. Confidence": pretty_metric(incorrect_df["Confidence"].mean()) if len(incorrect_df) > 0 else np.nan,
        "Avg. Leaf Focus Ratio": pretty_metric(incorrect_df["LeafFocusRatio"].mean()) if len(incorrect_df) > 0 else np.nan,
        "Avg. Background Leakage": pretty_metric(incorrect_df["BackgroundLeakage"].mean()) if len(incorrect_df) > 0 else np.nan,
        "Interpretation": "More dispersed or misleading attention patterns",
    })

table8_df = pd.DataFrame(table8_rows)

save_table(table8_df, "Table_8_Correct_vs_Incorrect_Explanation_Behavior")

print("Table 8 saved successfully")
display(table8_df)

Saved table: /kaggle/working/thesis_outputs/rq4_explainability/tables/Table_8_Correct_vs_Incorrect_Explanation_Behavior.csv
Table 8 saved successfully


,Model,Prediction Type,Avg. Confidence,Avg. Leaf Focus Ratio,Avg. Background Leakage,Interpretation
0,ResNet50,Correct Predictions,0.7622,0.8844,0.1156,More concentrated attention on relevant leaf r...
1,ResNet50,Incorrect Predictions,0.7010,0.8319,0.1681,More dispersed or misleading attention patterns
2,EfficientNet-B0,Correct Predictions,0.8358,0.9285,0.0715,More concentrated attention on relevant leaf r...
3,EfficientNet-B0,Incorrect Predictions,0.6930,0.8427,0.1573,More dispersed or misleading attention patterns
4,MobileNetV2,Correct Predictions,0.6272,0.8613,0.1387,More concentrated attention on relevant leaf r...
5,MobileNetV2,Incorrect Predictions,0.5594,0.8505,0.1495,More dispersed or misleading attention patterns


In [21]:
# ----------------------------------------
# Section 14: Figure 8 - Explanation behavior comparison
# ----------------------------------------

fig, ax = plt.subplots(figsize=(12, 5.5))

model_order = ["ResNet50", "EfficientNet-B0", "MobileNetV2"]

categories = [
    ("Avg. Confidence", "Correct Predictions", "Conf. (Correct)"),
    ("Avg. Confidence", "Incorrect Predictions", "Conf. (Incorrect)"),
    ("Avg. Leaf Focus Ratio", "Correct Predictions", "Leaf Focus (Correct)"),
    ("Avg. Leaf Focus Ratio", "Incorrect Predictions", "Leaf Focus (Incorrect)"),
    ("Avg. Background Leakage", "Correct Predictions", "Bg. Leakage (Correct)"),
    ("Avg. Background Leakage", "Incorrect Predictions", "Bg. Leakage (Incorrect)"),
]

x = np.arange(len(categories))
width = 0.23

for i, model_name in enumerate(model_order):
    values = []

    for metric_col, pred_type, _ in categories:
        row = table8_df[
            (table8_df["Model"] == model_name) &
            (table8_df["Prediction Type"] == pred_type)
        ].iloc[0]
        values.append(row[metric_col])

    ax.bar(x + (i - 1) * width, values, width=width, label=model_name)

ax.set_xticks(x)
ax.set_xticklabels([label for _, _, label in categories], rotation=18, ha="right")
ax.set_ylim(0, 1.0)
ax.set_ylabel("Score")
ax.set_title("Figure 8. Comparison of Explanation Behavior in Correct and Incorrect Predictions Across Backbone Models")
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=True)

save_figure(fig, "Figure_8_Explanation_Behavior_Comparison")

Saved figure: /kaggle/working/thesis_outputs/rq4_explainability/figures/Figure_8_Explanation_Behavior_Comparison.pdf


In [16]:
# ----------------------------------------
# Section 15: Save RQ4 metadata
# ----------------------------------------

meta_dir = ensure_dir(OUTPUT_ROOT / "metadata")

rq4_meta = {
    "seed": SEED,
    "models_evaluated": CONFIG["model_names"],
    "figure7_models": CONFIG["figure7_models"],
    "audit_batches": CONFIG["audit_batches"],
    "num_classes": NUM_CLASSES,
    "audited_samples_total": len(audit_df),
    "figure7_output_mode": "one_figure_per_model",
}

meta_path = meta_dir / "rq4_metadata.json"
with open(meta_path, "w") as f:
    json.dump(rq4_meta, f, indent=2)

print("RQ4 metadata saved successfully")
print(f"Metadata path: {meta_path}")

RQ4 metadata saved successfully
Metadata path: /kaggle/working/thesis_outputs/rq4_explainability/metadata/rq4_metadata.json


In [17]:
# ----------------------------------------
# Section 16: Create ZIP archive
# ----------------------------------------

zip_path = OUTPUT_ROOT.parent / "05_rq4_explainability_outputs.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in OUTPUT_ROOT.rglob("*"):
        if file_path.is_file():
            zf.write(file_path, arcname=file_path.relative_to(OUTPUT_ROOT))

print("ZIP archive created successfully")
print(f"ZIP file: {zip_path}")
print("05_rq4_explainability notebook completed successfully")

ZIP archive created successfully
ZIP file: /kaggle/working/thesis_outputs/05_rq4_explainability_outputs.zip
05_rq4_explainability notebook completed successfully
